In [ ]:
from scipy.stats import invgamma

import numpy as np
import torch
from abc import ABC, abstractmethod
import numpy as np
import torch
from torch import autograd, nn
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Callable, Literal, overload
from tqdm import tqdm
import random
import pandas as pd
import os
import kagglehub
# import kaggle

In [ ]:
SEED_DEFAULT = 34
sns.set_theme()
torch.manual_seed(SEED_DEFAULT)
rng = np.random.default_rng(SEED_DEFAULT)
 # Python
random.seed(SEED_DEFAULT)

# NumPy
np.random.seed(SEED_DEFAULT)

# PyTorch CPU
torch.manual_seed(SEED_DEFAULT)

# PyTorch CUDA
torch.cuda.manual_seed(SEED_DEFAULT)
torch.cuda.manual_seed_all(SEED_DEFAULT)

# Forçar determinismo
torch.use_deterministic_algorithms(True)

# Evitar fontes não determinísticas no CUDA
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
path = kagglehub.dataset_download("zakariaeyoussefi/song-year-prediction-msd")

complete_path = os.path.join(path, "YearPredictionMSD.csv")
df_yearMSD = pd.read_csv(complete_path)

# Convertendo dados para tensores
columns_features = [c for c in df_yearMSD.columns if c != "Year"]

X = df_yearMSD[columns_features].values   # shape (N, d)
y = df_yearMSD["Year"].values             # shape (N,)

Using Colab cache for faster access to the 'song-year-prediction-msd' dataset.


In [ ]:
# shapes
N, d = X.shape

# tensores
X_tensor = torch.from_numpy(X).to(dtype=torch.float64)
y_tensor = torch.from_numpy(y).to(dtype=torch.float64)

# ---------- normalização de X ----------
X_mean = X_tensor.mean(dim=0, keepdim=True)
X_std  = X_tensor.std(dim=0, keepdim=True, unbiased=False)

# evita divisão por zero (features constantes)
X_std = torch.clamp(X_std, min=1e-8)

X_norm = (X_tensor - X_mean) / X_std   # (N, d)

# ---------- normalização de y ----------
y_mean = y_tensor.mean()
y_std  = y_tensor.std(unbiased=False)
y_std  = torch.clamp(y_std, min=1e-8)

y_norm = (y_tensor - y_mean) / y_std   # (N,)

# ---------- intercept ----------
intercept = torch.ones((N, 1), dtype=torch.float64)

# ---------- X com intercept ----------
X_with_intercept = torch.cat([intercept, X_norm], dim=1)  # (N, d+1)

# ---------- Xy final ----------
Xy = torch.cat([X_with_intercept, y_norm.unsqueeze(1)], dim=1)  # (N, d+2)


In [ ]:
Xy[:2, :]

tensor([[ 1.0000,  1.0806,  0.3913,  1.8265,  0.4647, -0.4747, -0.2782, -1.5524,
         -1.3108,  0.3877, -0.6662,  0.7934, -0.5843, -1.0561, -1.0451, -0.8059,
         -0.7474, -1.0553, -0.8588, -0.8731, -0.9034, -0.6660, -0.8362, -1.0080,
         -0.7348, -0.4237, -0.5046,  0.2612,  0.3470, -0.6778, -0.4639, -0.0319,
          0.1447,  0.0299,  0.1036,  0.1717, -0.6767, -0.1981, -0.4437,  0.5854,
          0.2428, -0.3011, -0.1776,  0.3768, -0.4290,  0.4195, -0.4536,  0.0073,
          0.3731,  0.3638,  0.0519, -0.3395, -0.4291,  0.0074,  0.4785,  0.0509,
         -0.3108,  0.0022,  0.2411, -0.0745, -0.1152, -0.1953,  0.1551, -0.2723,
          0.1388, -0.3661, -0.2796,  0.0154,  0.3712, -0.0351,  0.1863, -0.1121,
         -0.2007,  0.1156,  0.3024,  0.2005, -0.0126,  0.0409, -0.1139,  0.2518,
          0.1065, -0.0853,  0.1085,  0.1428, -0.2374,  0.0492, -0.3562,  0.5445,
         -0.4706, -0.2560,  0.0423,  0.2381],
        [ 1.0000,  0.8809,  0.3323,  1.7485,  0.7218, -0.1649, 

In [ ]:
Xy[:, -1].unique()

tensor([-6.9890, -6.8060, -6.7146, -6.6231, -6.5316, -6.4401, -6.3486, -6.2571,
        -6.1657, -6.0742, -5.9827, -5.8912, -5.7997, -5.7082, -5.6168, -5.5253,
        -5.4338, -5.3423, -5.2508, -5.1594, -5.0679, -4.9764, -4.8849, -4.7934,
        -4.7019, -4.6105, -4.5190, -4.4275, -4.3360, -4.2445, -4.1530, -4.0616,
        -3.9701, -3.8786, -3.7871, -3.6956, -3.6041, -3.5127, -3.4212, -3.3297,
        -3.2382, -3.1467, -3.0553, -2.9638, -2.8723, -2.7808, -2.6893, -2.5978,
        -2.5064, -2.4149, -2.3234, -2.2319, -2.1404, -2.0489, -1.9575, -1.8660,
        -1.7745, -1.6830, -1.5915, -1.5000, -1.4086, -1.3171, -1.2256, -1.1341,
        -1.0426, -0.9512, -0.8597, -0.7682, -0.6767, -0.5852, -0.4937, -0.4023,
        -0.3108, -0.2193, -0.1278, -0.0363,  0.0552,  0.1466,  0.2381,  0.3296,
         0.4211,  0.5126,  0.6041,  0.6955,  0.7870,  0.8785,  0.9700,  1.0615,
         1.1529], dtype=torch.float64)

In [ ]:
# @title Sampler
class Sampler:
    def __init__(self, f_log_prob: Callable |None = None, device = None, dtype = torch.float64):
        self.__f_log_prob = f_log_prob
        self.amostras = None
        self.__device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.__dtype = dtype

    @property
    def f_log_prob(self):
        return self.__f_log_prob

    @f_log_prob.setter
    def f_log_prob(self, f_log_prob):
        self.__f_log_prob = f_log_prob

    def __mh_updater(self, estado_atual: torch.Tensor, logp_atual: float, use_log_pdf: bool, f_next_state: Callable|None = None):
        was_scalar = False
        if estado_atual.ndim == 0:
            estado_atual = estado_atual.unsqueeze(-1)
            was_scalar = True
        if not f_next_state:
            dim = estado_atual.shape[-1]
            cov = torch.eye(dim, device=estado_atual.device, dtype=estado_atual.dtype) * 0.24
            f_next_state = lambda x: torch.distributions.MultivariateNormal(x, covariance_matrix=cov).sample()
        estado_proposto = f_next_state(estado_atual)

        # verossimilhanca
        logp_proposto = self.__f_log_prob(estado_proposto)
        # threshold atual
        threshold = torch.rand((), device=self.__device, dtype=self.__dtype)
        if use_log_pdf:
            threshold = threshold.log()

            # # --- > if (log(verossimilhanca_propost / verossimilhanca_atual)) > log(threshcold)
            # # --- > if (log(verossimilhanca_propost) - log(verossimilhanca_atual)) > log(threshcold
            # print("proposto: ", logp_proposto)
            # print("atual: ", logp_atual)
            # print(f"threshold {threshold} < {logp_proposto - logp_atual} proposto - atual ==>  ", threshold < logp_proposto - logp_atual)
            # print(threshold)
            # print("-"*25)
            # print(logp_proposto)
            # print("proposto: ", logp_proposto)
            # print("Atual: ", logp_atual)
            if threshold < logp_proposto - logp_atual:
                return estado_proposto, estado_proposto, logp_proposto, True

            return estado_atual, estado_proposto, logp_atual, False

        aceitacao_raio = logp_proposto / logp_atual
        if threshold < min(aceitacao_raio, 1):
            return estado_proposto, estado_proposto, logp_proposto, True

        return estado_atual, estado_proposto, logp_atual, False

    def __hmc_updater(self, estado_atual: torch.Tensor, logp_atual: float, step_size, n_steps: int, f_grad_log_prob: Callable):
        r0 = torch.randn_like(estado_atual, device= self.__device, dtype=self.__dtype)

        valor_inicial = logp_atual - 0.5 * torch.sum(r0**2)

        #leapfrog_integration
        # https://www.tcbegley.com/blog/posts/mcmc-part-2
        theta = estado_atual.clone()
        r = r0.clone()

        r = r + 0.5 * step_size * f_grad_log_prob(theta)
        for _ in range(n_steps):
            theta = theta + step_size * r

            grad = f_grad_log_prob(theta)
            if _ != n_steps - 1:
                r = r + step_size * grad

        r = r + 0.5 * step_size * grad
        r = -r

        logp_proposto = self.__f_log_prob(theta)

        valor_final = logp_proposto - 0.5 * torch.sum(r ** 2)

        log_accept_ratio = valor_final - valor_inicial
        threshold = torch.rand((), device=self.__device, dtype=self.__dtype).log()
        if threshold < log_accept_ratio:
            return theta, theta, logp_proposto, True

        return estado_atual, theta, logp_atual, False

    @overload
    def get_samples(self, method: Literal["mh"], estado_inicial: torch.Tensor, use_log_pdf: bool, n_amostras: int = 1000, burnin: float = 0.2, f_next_state: Callable|None = None) -> torch.Tensor: ...

    @overload
    def get_samples(self, method: Literal["hmc"], estado_inicial: torch.Tensor, step_size, n_steps: int, f_grad_log_prob: Callable, n_amostras: int = 1000, burnin: float = 0.2) -> torch.Tensor: ...

    def get_samples(self, method: str, estado_inicial: torch.Tensor, n_amostras: int=1000, burnin: float=0.2, **kwargs):
        if method.lower() == "mh":
            updater = self.__mh_updater
        elif method.lower() == "hmc":
            updater = self.__hmc_updater
        else:
            raise ValueError("method deve ser 'hmc' ou 'mh'")

        if self.__f_log_prob is None:
            raise ValueError("f_log_prob é um argumento necessário para a execução de qualquer método")

        estado_atual = torch.as_tensor(estado_inicial, device=self.__device, dtype=self.__dtype)
        logp_atual = self.__f_log_prob(estado_atual)

        burnin_amostras = int(burnin * n_amostras)
        amostras = []
        burned = []
        not_accepted = []

        for _ in tqdm(range(n_amostras), desc="Gerando amostras"):
            estado_atual, estado_proposto, logp_atual, foi_aceito = updater(estado_atual, logp_atual, **kwargs)


            if _ > burnin_amostras:
                if not foi_aceito:
                    not_accepted.append(estado_proposto.clone())
                else:
                    amostras.append(estado_atual.clone())
            else:
                burned.append(estado_atual.clone())

        self.amostras = torch.stack(amostras)
        self.burned = torch.stack(burned)
        self.not_accepted = torch.stack(not_accepted)
        return self.amostras, self.burned, self.not_accepted

In [ ]:
# @title distribuições base

class Distribution(ABC):
    @abstractmethod
    def pdf(self, x: torch.Tensor) -> torch.Tensor: ...

    @abstractmethod
    def log(self, x: torch.Tensor) -> torch.Tensor: ...

# TODO: criar laplace
class MultivariateLaplaceL1(Distribution):
    def __init__(self, mean, b, device=None, dtype=torch.float64, eps=1e-12):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = dtype
        self.eps = eps

        self.mean = torch.as_tensor(mean, dtype=dtype, device=self.device)  # (d,)
        self.b = torch.as_tensor(b, dtype=dtype, device=self.device)        # escalar ou (d,)
        if self.b.ndim == 0:
            self.b = self.b.expand_as(self.mean)
        if self.b.shape != self.mean.shape:
            raise ValueError("b deve ser escalar ou ter shape (d,)")

        if torch.any(self.b <= 0):
            raise ValueError("b precisa ser > 0")

        self.d = self.mean.numel()
        self.logZ = torch.sum(torch.log(2 * self.b))  # sum_i log(2 b_i)

    def pdf(self, x):
        x = torch.as_tensor(x, dtype=self.dtype, device=self.device)
        diff = x - self.mean
        logp = -torch.sum(torch.abs(diff) / self.b, dim=-1) - self.logZ
        return torch.exp(logp)

    def log(self, x):
        x = torch.as_tensor(x, dtype=self.dtype, device=self.device)
        diff = x - self.mean
        return -torch.sum(torch.abs(diff) / self.b, dim=-1) - self.logZ

    def grad_log(self, x):
        x = torch.as_tensor(x, dtype=self.dtype, device=self.device)
        diff = x - self.mean
        # sign suave pra evitar NaN em 0
        sign = diff / torch.sqrt(diff * diff + self.eps)
        return -(sign / self.b)

    def sample(self, size=1):
        dist = torch.distributions.Laplace(loc=self.mean, scale=self.b)
        return dist.sample((size,))


# TODO: criar normal multivariate
class MultivariateNormal(Distribution):
    def __init__(
        self,
        mean,
        cov,
        device=None,
        dtype=torch.float64,
        eps: float = 1e-12,
    ):
        """
        mean: tensor shape (d,)
        cov:  tensor shape (d,d), simétrica e definida positiva
        """
        self.__device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.__dtype = dtype
        self.__eps = eps

        self.__mean = torch.as_tensor(mean, dtype=dtype, device=self.__device)
        self.__cov = torch.as_tensor(cov, dtype=dtype, device=self.__device)

        if self.__mean.ndim != 1:
            raise ValueError("mean precisa ter shape (d,)")
        if self.__cov.ndim != 2 or self.__cov.shape[0] != self.__cov.shape[1]:
            raise ValueError("cov precisa ter shape (d,d)")
        if self.__cov.shape[0] != self.__mean.shape[0]:
            raise ValueError("dimensões de mean e cov não batem")

        self.__d = self.__mean.shape[0]

        # Pré-computações
        self.__cov = self.__cov + eps * torch.eye(self.__d, device=self.__device, dtype=self.__dtype)
        self.__inv_cov = torch.linalg.inv(self.__cov)
        self.__logdet_cov = torch.logdet(self.__cov)

        self.__logZ = self.__d * torch.log(torch.tensor(2 * torch.pi, dtype=dtype, device=self.__device)) \
                      + self.__logdet_cov

    # -------- pdf --------
    def pdf(self, x: torch.Tensor):
        """
        x: shape (..., d)
        retorna: shape (...)
        """
        x = torch.as_tensor(x, dtype=self.__dtype, device=self.__device)
        diff = x - self.__mean
        quad = torch.einsum("...i,ij,...j->...", diff, self.__inv_cov, diff)
        return torch.exp(-0.5 * quad) / torch.exp(0.5 * self.__logZ)

    # -------- log pdf --------
    def log(self, x: torch.Tensor):
        """
        log p(x)
        """
        x = torch.as_tensor(x, dtype=self.__dtype, device=self.__device)
        diff = x - self.__mean
        quad = torch.einsum("...i,ij,...j->...", diff, self.__inv_cov, diff)
        return -0.5 * (quad + self.__logZ)

    # -------- gradiente do log (score) --------
    def grad_log(self, x: torch.Tensor):
        """
        ∇_x log p(x) = - Σ^{-1} (x - μ)
        retorna shape (..., d)
        """
        x = torch.as_tensor(x, dtype=self.__dtype, device=self.__device)
        diff = x - self.__mean
        grad = -torch.einsum("ij,...j->...i", self.__inv_cov, diff)
        return grad

    # -------- hessiana do log --------
    def hessian_log(self, x: torch.Tensor):
        """
        ∇²_x log p(x) = - Σ^{-1}
        retorna shape (..., d, d) ou (d,d) se x for 1D
        """
        if x.ndim == 1:
            return -self.__inv_cov

        batch_shape = x.shape[:-1]
        return -self.__inv_cov.expand(*batch_shape, self.__d, self.__d)

    # -------- amostragem --------
    def sample(self, size=1):
        """
        retorna shape (size, d)
        """
        dist = torch.distributions.MultivariateNormal(self.__mean, self.__cov)
        return dist.sample((size,)).to(dtype=self.__dtype, device=self.__device)


In [ ]:
# @title data density
class NormalRegressaoVarianciaConhecida:
    def __init__(self, sigma2: float | int, device=None, dtype=torch.float64, eps: float = 1e-12):
        """
        Modelo:
          y | x, theta ~ Normal(x^T theta, sigma2)

        Entrada Xy:
          Xy[:-1] = x (vetor)
          Xy[-1]  = y (escalar)

        sigma2: variância conhecida (escala é sigma = sqrt(sigma2))
        """
        self.__device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.__dtype = dtype
        self.__eps = float(eps)

        self.__sigma2 = torch.as_tensor(sigma2, dtype=self.__dtype, device=self.__device)
        if torch.any(self.__sigma2 <= 0):
            raise ValueError("sigma2 precisa ser > 0")

        self.__log_norm = 0.5 * torch.log(2 * torch.pi * self.__sigma2)  # 0.5*log(2πσ²)

    def _split(self, Xy, theta):
        Xy = torch.as_tensor(Xy, dtype=self.__dtype, device=self.__device)
        theta = torch.as_tensor(theta, dtype=self.__dtype, device=self.__device)

        if Xy.ndim == 1:
            x = Xy[:-1]
            y = Xy[-1]
            # print(x)
            # print(x.shape)
            # print(theta)
            # print(theta.shape)
            mu = torch.dot(x, theta)
        else:
            # print("AQYU!!")
            x = Xy[..., :-1]              # (..., d)
            y = Xy[..., -1]               # (...)
            mu = torch.einsum("...i,i->...", x, theta)  # (...)

        return x, y, mu

    # ---------- pdf ----------
    def pdf(self, Xy, theta):
        _, y, mu = self._split(Xy, theta)
        r = y - mu
        return torch.exp(-0.5 * (r * r) / self.__sigma2) / torch.sqrt(2 * torch.pi * self.__sigma2)

    # ---------- log pdf ----------
    def log(self, Xy, theta):
        _, y, mu = self._split(Xy, theta)
        r = y - mu
        return -(0.5 * (r * r) / self.__sigma2) - self.__log_norm

    # ---------- gradiente do log em relação a theta ----------
    def grad_log_theta(self, Xy, theta):
        """
        ∇_theta log p(y|x,theta) = (1/sigma2) * (y - x^T theta) * x
        Retorna:
          - se Xy é (d+1,), retorna (d,)
          - se Xy é (..., d+1), retorna (..., d)
        """
        x, y, mu = self._split(Xy, theta)
        r = y - mu

        if x.ndim == 1:
            return (r / self.__sigma2) * x
        return (r[..., None] / self.__sigma2) * x

    # ---------- hessiana do log em relação a theta ----------
    def hessian_log_theta(self, Xy, theta=None):
        """
        ∇²_theta log p(y|x,theta) = -(1/sigma2) * x x^T
        (independe de theta)
        Retorna:
          - se Xy é (d+1,), retorna (d,d)
          - se Xy é (..., d+1), retorna (..., d, d)
        """
        x = torch.as_tensor(Xy, dtype=self.__dtype, device=self.__device)
        x = x[:-1] if x.ndim == 1 else x[..., :-1]

        if x.ndim == 1:
            return -(1.0 / self.__sigma2) * torch.outer(x, x)

        return -(1.0 / self.__sigma2) * torch.einsum("...i,...j->...ij", x, x)


In [ ]:
# @title  Auxiliar conjugada to sample
def sample_theta_conjugate(
    X: torch.Tensor,          # (N, d)
    y: torch.Tensor,          # (N,)
    sigma2: float,
    n_samples: int,
    device=None,
    dtype=torch.float64,
    eps: float = 1e-8,
):
    """
    Gera amostras do posterior conjugado:
      theta | X,y ~ N(mu_n, Sigma_n)
    com prior N(0, I)
    """
    device = device or X.device

    X = X.to(device=device, dtype=dtype)
    y = y.to(device=device, dtype=dtype)

    N, d = X.shape
    sigma2 = torch.tensor(sigma2, dtype=dtype, device=device)

    # Posterior covariance
    XtX = X.T @ X                                # (d,d)
    precision = torch.eye(d, device=device, dtype=dtype) + XtX / sigma2
    Sigma = torch.linalg.inv(precision + eps * torch.eye(d, device=device, dtype=dtype))

    # Posterior mean
    mu = Sigma @ (X.T @ y) / sigma2              # (d,)

    # Amostragem
    dist = torch.distributions.MultivariateNormal(mu, Sigma)
    theta_samples = dist.sample((n_samples,))    # (Tf, d)

    return theta_samples, mu, Sigma

In [ ]:
#@title sampling theta_f
# dimensão do theta
d_plus_1 = X_with_intercept.shape[1]

# theta = torch.zeros(d_plus_1, dtype=torch.float64)

theta_hat = torch.linalg.lstsq(X_with_intercept, y_norm).solution
res = y_norm - X_with_intercept @ theta_hat
sigma2_hat = res.pow(2).mean()
model = NormalRegressaoVarianciaConhecida(sigma2=sigma2_hat)

# Extrair X e y
X = Xy[:, :-1]   # (N, d)
y = Xy[:, -1]    # (N,)

# Gerar amostras do falso posterior
Tf = 2000
theta_samples, theta_mean, theta_cov = sample_theta_conjugate(
    X=X,
    y=y,
    sigma2=sigma2_hat,
    n_samples=Tf,
    dtype=torch.float64
)

print(theta_samples[:1, :])
print(theta_samples.shape)  # (Tf, d)

/tmp/ipython-input-132762640.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sigma2 = torch.tensor(sigma2, dtype=dtype, device=device)


tensor([[ 3.1385e-03,  4.8303e-01, -2.6829e-01, -1.3869e-01,  1.9472e-03,
         -3.1514e-02, -2.5846e-01, -8.5844e-03, -7.2132e-02, -6.9100e-02,
          1.7126e-02, -6.5160e-02, -1.2955e-03,  9.7498e-02,  5.2848e-02,
         -5.0198e-02,  5.7242e-02,  2.1135e-02,  7.3832e-02,  5.5642e-02,
          5.7861e-02,  1.2418e-02,  5.0751e-04,  1.3250e-01,  4.1621e-02,
         -3.7602e-02,  4.2772e-03,  7.9545e-02,  8.6149e-03,  1.5156e-02,
         -6.7826e-03, -1.0870e-02, -9.8905e-03, -4.0176e-02,  1.2195e-02,
          4.8826e-03, -5.1548e-02, -8.4328e-03,  2.6393e-02,  3.2486e-02,
         -2.9662e-02, -2.0120e-02, -8.5814e-03, -8.6290e-03, -7.4298e-03,
         -1.3702e-02,  3.2060e-02,  1.9341e-02, -4.9636e-02,  6.2829e-03,
          2.1401e-02,  4.0694e-03, -1.2887e-02,  1.1893e-02,  3.1250e-03,
          2.1575e-03,  5.2291e-03, -5.6594e-02,  5.0041e-02, -2.1865e-02,
          1.8464e-03, -1.8684e-02, -8.5748e-03, -2.5872e-02,  3.4137e-02,
         -4.2675e-02,  9.0496e-03, -4.

In [ ]:
# @title score matching loss
def score_matching_loss_regression_alpha(
    theta_samples: torch.Tensor,   # (Tf, d)
    alpha_Xy: torch.Tensor,        # (k, d+1)  [x..., y]
    n: int,
    sigma2: float | torch.Tensor,
    eps: float = 1e-12,
):
    """
    Hyvarinen score matching loss para:
      log \tilde p_f^alpha(theta) = log N(0,I)(theta) + (n/k) * sum_j log N(y_j | x_j^T theta, sigma2)

    Retorna escalar: média sobre k (agrega pseudo-dados) e sobre Tf.
    """
    theta = torch.as_tensor(theta_samples)
    alpha = alpha_Xy.to(dtype=theta.dtype, device=theta.device)
    sigma2 = torch.as_tensor(sigma2, dtype=theta.dtype, device=theta.device)
    sigma2 = torch.clamp(sigma2, min=eps)

    Tf, d = theta.shape
    k = alpha.shape[0]
    # print(alpha.shape[1], d+1)
    assert alpha.shape[1] == d + 1, f"alpha_Xy deve ser (k, d+1). Recebi {alpha.shape} e d={d}"

    X = alpha[:, :-1]        # (k, d)
    y = alpha[:, -1]         # (k,)

    # resíduos para cada (j,t): r_{j,t} = y_j - x_j^T theta_t
    # X @ theta^T => (k, Tf)
    mu = X @ theta.T                         # (k, Tf)
    r = y.unsqueeze(1) - mu                  # (k, Tf)

    scale = (n / k) / sigma2                 # escalar (ou tensor)
    # score s(theta) = ∇_theta log prior + (n/k)*Σ_j ∇ loglik_j
    # ∇ log prior N(0,I) = -theta
    # ∇ loglik_j = (1/sigma2) * r * x_j
    # somando em j: Σ_j (r_{j,t} x_j) = (r^T @ X)   com r: (k,Tf) => r^T: (Tf,k) => (Tf,d)
    sum_rX = r.T @ X                         # (Tf, d)
    s = -theta + scale * sum_rX              # (Tf, d)

    # Laplaciana Δ log \tilde p = tr(H)
    # H_prior = -I  => tr = -d
    # H_lik_j = -(1/sigma2) x_j x_j^T  => tr = -(1/sigma2)||x_j||^2
    # com fator (n/k) e soma em j:
    # tr_total = -d - (n/k)/sigma2 * Σ_j ||x_j||^2
    sum_x2 = torch.sum(X * X)                # escalar = Σ_j ||x_j||^2
    lap = -d - scale * sum_x2                # escalar (independe de theta)

    # Hyvarinen: E[ 0.5||s||^2 + Δ log p ]
    loss = 0.5 * torch.mean(torch.sum(s * s, dim=1)) + lap
    return loss

def init_alpha_stratified_by_y(Xy: torch.Tensor, k: int, n_bins: int = 16):
    Xy = torch.as_tensor(Xy)
    y = Xy[:, -1]
    N = Xy.shape[0]

    # bins por quantis
    qs = torch.linspace(0, 1, n_bins + 1, device=Xy.device, dtype=Xy.dtype)
    edges = torch.quantile(y, qs)

    idxs = []
    per_bin = max(1, k // n_bins)

    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        mask = (y >= lo) & (y <= hi) if b == n_bins - 1 else (y >= lo) & (y < hi)
        candidates = torch.where(mask)[0]
        if candidates.numel() == 0:
            continue
        take = min(per_bin, candidates.numel())
        chosen = candidates[torch.randperm(candidates.numel(), device=Xy.device)[:take]]
        idxs.append(chosen)

    idx = torch.cat(idxs)
    if idx.numel() > k:
        idx = idx[torch.randperm(idx.numel(), device=Xy.device)[:k]]
    elif idx.numel() < k:
        # completa com aleatório
        extra = torch.randperm(N, device=Xy.device)[: (k - idx.numel())]
        idx = torch.cat([idx, extra])

    return Xy[idx].clone()


# @title find alpha star
def fit_alpha_with_adam(
    theta_samples: torch.Tensor,   # (Tf,d)
    Xy,
    n: int,
    k: int,
    sigma2: float,
    steps: int = 300,
    lr: float = 0.05,
    init_y: torch.Tensor | None = None,
):
    # # X = torch.as_tensor(X_alpha_fixed, dtype=theta.dtype, device=theta.device)

    # # k, d = X.shape
    # # if init_y is None:
    # #     y_alpha = 0.05 * torch.randn(k, device=theta.device, dtype=theta.dtype)
    # # else:
    # #     y_alpha = torch.as_tensor(init_y, device=theta.device, dtype=theta.dtype).clone()

    # # y_alpha.requires_grad_()

    # # opt = torch.optim.Adam([y_alpha], lr=lr)
    theta = torch.as_tensor(theta_samples)
    d = theta.shape[1]
    # alphas = torch.randn((k, d + 1), dtype=theta.dtype, device=theta.device, requires_grad=True)
    alphas = init_alpha_stratified_by_y(Xy, k=256, n_bins=16)
    alphas.requires_grad_()
    opt = torch.optim.Adam([alphas], lr=lr)

    for it in range(steps):
        opt.zero_grad()
        # loss = 0
        # for idx, alpha in enumerate(alphas):
        loss = score_matching_loss_regression_alpha(theta, alphas, n=n, sigma2=sigma2)
        loss = loss + 1e-2 * (alphas[:, :-1]**2).mean() + 1e-2 * (alphas[:, -1]**2).mean()

        loss.backward()
        opt.step()

        if it % 50 == 0 or it == steps - 1:
            print(f"it={it} loss={loss.item():.6f}")

    return alphas.detach()

In [ ]:
len_alpha = 256
alpha_star = fit_alpha_with_adam(
    theta_samples=theta_samples,
    Xy=Xy,
    n=X.shape[0],          # n do paper
    k=256,
    sigma2=sigma2_hat,
    steps=10000,
    lr=0.0005
)

it=0 loss=75668392873.448166
it=50 loss=33524503955.462742
it=100 loss=14160036927.978207
it=150 loss=5523132136.231700
it=200 loss=2025211160.821459
it=250 loss=702089756.290922
it=300 loss=224806802.840355
it=350 loss=64617713.395630
it=400 loss=15679297.586951
it=450 loss=2124440.863822
it=500 loss=-1301492.272623
it=550 loss=-2105148.236255
it=600 loss=-2291630.426820
it=650 loss=-2346236.751066
it=700 loss=-2375881.518814
it=750 loss=-2401996.520393
it=800 loss=-2428468.909551
it=850 loss=-2455926.378345
it=900 loss=-2484452.834969
it=950 loss=-2514052.159314
it=1000 loss=-2544718.666624
it=1050 loss=-2576446.445262
it=1100 loss=-2609230.328517
it=1150 loss=-2643065.881544
it=1200 loss=-2677949.302422
it=1250 loss=-2713877.329452
it=1300 loss=-2750847.160562
it=1350 loss=-2788856.383257
it=1400 loss=-2827902.913375
it=1450 loss=-2867984.941250
it=1500 loss=-2909100.884144
it=1550 loss=-2951249.344026
it=1600 loss=-2994429.069952
it=1650 loss=-3038638.924430
it=1700 loss=-3083877.8

In [ ]:
alpha_star

tensor([[ 0.8975, -0.7297,  1.1719,  ..., -0.4801, -1.9592, -2.2635],
        [ 0.9001, -1.3126, -0.7296,  ..., -2.1688, -1.0180, -2.6287],
        [ 0.8999, -0.5297, -0.4052,  ...,  0.5771, -0.4830, -3.7184],
        ...,
        [ 1.0803,  1.3464,  0.6572,  ..., -0.7222,  0.0233,  0.9376],
        [ 1.0567, -0.2852,  0.2624,  ...,  0.1657, -0.4767,  0.9243],
        [ 1.0540,  0.8696, -0.0924,  ..., -0.2126,  1.6521,  0.9011]],
       dtype=torch.float64)

In [ ]:
alpha_star[:, -1].unique()

tensor([-6.4424, -5.1892, -4.1766, -3.8781, -3.8218, -3.7184, -3.1673, -2.8507,
        -2.6287, -2.4054, -2.2635, -2.2383, -2.1706, -2.0795, -2.0795, -2.0697,
        -2.0471, -1.9663, -1.9020, -1.8901, -1.7110, -1.6363, -1.6288, -1.5909,
        -1.5267, -1.4967, -1.4918, -1.4095, -1.3887, -1.3854, -1.3442, -1.2367,
        -1.2092, -1.1921, -1.1234, -1.1213, -1.1082, -1.1070, -1.0623, -1.0118,
        -1.0018, -0.9559, -0.9187, -0.8794, -0.8207, -0.8110, -0.7868, -0.7756,
        -0.7693, -0.7474, -0.7056, -0.6954, -0.6847, -0.6432, -0.6379, -0.6025,
        -0.5878, -0.5813, -0.5715, -0.5668, -0.5655, -0.4746, -0.4724, -0.4464,
        -0.4399, -0.4397, -0.4324, -0.4280, -0.3536, -0.3330, -0.3192, -0.3156,
        -0.3004, -0.2934, -0.2314, -0.2280, -0.2021, -0.1730, -0.1711, -0.1477,
        -0.1387, -0.1377, -0.1321, -0.0981, -0.0944, -0.0919, -0.0876, -0.0768,
        -0.0721, -0.0719, -0.0557, -0.0426, -0.0335, -0.0305, -0.0130, -0.0099,
        -0.0098,  0.0183,  0.0263,  0.02

In [ ]:
# @title definindo distribuições
d = theta_samples.shape[1]
n_data = X.shape[0]
# print(d, n_data)

mean = torch.zeros(d, dtype=theta_samples.dtype, device=theta_samples.device)

# Normal pi fake
sig = torch.full((d,), 2.0, dtype=mean.dtype, device=mean.device)
sig[0] = 5.0  # intercept (se theta[0] for intercept)

cov_f = torch.diag(sig**2)
pi_f = MultivariateNormal(mean=mean, cov=cov_f)

# Laplace pi target
b = torch.ones(d, dtype=mean.dtype, device=mean.device)
b[0] = 5.0      # intercept solto
b[1:] = 1.0     # coeficientes

pi = MultivariateLaplaceL1(mean=mean, b=b)

# distribuição dos dados real
data_distribution = NormalRegressaoVarianciaConhecida(sigma2 = sigma2_hat)

In [ ]:
# @title parametric distribution

class ParametricDistribution:
    def __init__(self, target_priori, fake_priori, data_distribution, alpha_star, n, k, dtype=torch.float64) -> None:
        self.__target_priori = target_priori
        self.__fake_priori = fake_priori
        self.__data_distribution = data_distribution
        self.__dtype = dtype
        self.__alpha_star = alpha_star
        self.__rate = n / k

    def pdf_f(self, theta):
        data_evaluation_alpha = self.__data_distribution.pdf(self.__alpha_star, theta)
        prod_alphas = torch.prod(data_evaluation_alpha, dim=-1) ** self.__rate
        return self.__fake_priori.pdf(theta) * prod_alphas

    def log_f(self, theta):
        data_evaluation_alpha = self.__data_distribution.log(self.__alpha_star, theta)
        prod_alphas = torch.sum(data_evaluation_alpha, dim=-1) * self.__rate
        return self.__fake_priori.log(theta) + prod_alphas

    def pdf_s(self, theta):
        data_evaluation_alpha = self.__data_distribution.pdf(self.__alpha_star, theta)
        prod_alphas = torch.prod(data_evaluation_alpha, dim=-1) ** self.__rate
        return self.__target_priori.pdf(theta) * prod_alphas

    def log_s(self, theta):
        for alpha in self.__alpha_star:
            min_evaluation = self.__data_distribution.log(alpha, theta)
            # print(min_evaluation)
        data_evaluation_alpha = self.__data_distribution.log(self.__alpha_star, theta)
        prod_alphas = torch.sum(data_evaluation_alpha, dim=-1) * self.__rate
        return self.__target_priori.log(theta) + prod_alphas

    def grad_log_s(self, theta):
        data_evaluation_alpha = self.__data_distribution.grad_log_variancia(self.__alpha_star, theta)
        prod_alphas = torch.sum(data_evaluation_alpha, dim=-1) * self.__rate
        return self.__target_priori.grad_log(theta) + prod_alphas

    def get_weight(self, thetas, pf_data):
        pf = self.pdf_f(thetas)
        pf_data_eval = pf_data(thetas)
        weights = pf_data_eval / pf
        weights = weights / torch.sum(weights)

        return weights

In [ ]:
parametric_density = ParametricDistribution(
    target_priori=pi,
    fake_priori=pi_f,
    data_distribution=data_distribution,  # sua NormalRegressaoVarianciaConhecida(sigma2)
    alpha_star=alpha_star,                # (k, d+1)
    n=n_data,
    k=256 # FIXME: mudar se eu mudar o len_alpha
)

In [ ]:
# TODO: gerar as amostras de theta usando Sampler
sampler_ps = Sampler(f_log_prob=parametric_density.log_s)
estado_inicial = torch.randn((d,))
samples_ps, burned, not_used = sampler_ps.get_samples(
    method="mh",
    estado_inicial=estado_inicial,
    use_log_pdf=True,
    n_amostras=1000,
    burnin=0.1
)

Gerando amostras: 100%|██████████| 1000/1000 [00:12<00:00, 77.12it/s]


In [ ]:
samples_ps.shape

torch.Size([11, 91])

In [ ]:
samples_ps.mean(dim=0)

tensor([-0.9870, -1.3123, -0.6239,  0.6091,  0.9753,  0.7255, -1.1305,  1.8594,
        -0.1572,  1.4610, -0.3207, -0.7065,  1.5137,  0.4906, -0.4336, -1.7155,
        -2.4557,  0.3940, -0.3799, -0.0676,  1.0034,  2.2351,  1.4824,  1.3739,
        -0.3370, -0.2321,  0.2340, -0.2210, -1.9470,  0.6377, -0.3358, -0.0615,
         1.2536,  0.6067, -1.1244,  0.7531,  0.3592,  1.4479,  1.6926, -1.9655,
        -0.0365, -0.2485, -0.0846,  1.6448,  0.3476, -2.0405,  0.4229,  0.8044,
        -1.9340,  0.6395, -1.9557,  1.1813,  0.3000, -1.2945, -0.3689, -0.4533,
         0.4009, -0.6257,  1.4386, -1.5123,  1.3551, -2.0648, -1.3357, -0.5800,
         0.3658, -0.5594, -1.2317,  0.8775,  0.8044,  0.7349, -1.3214,  1.0818,
        -0.0213, -0.5621, -0.9770,  0.4123,  1.0812,  0.6833, -0.1660, -0.5386,
         1.8931, -2.1521,  0.7744, -1.2594,  1.5264,  0.5132,  0.2245,  0.4558,
        -0.5988,  0.7493,  0.8397], dtype=torch.float64)

In [ ]:
# @title Semi Parametric distribution (Neiswanger et al. Eq. 9-11)
class SemiParametricDistribution:
    def __init__(self, parametric_density: ParametricDistribution, priori_target, priori_fake, thetas, bandwidth):
        if thetas.ndim == 1:
            thetas = thetas.reshape(-1, 1)

        self.__num_fake_samples, self.__theta_dimension = thetas.shape
        self.__thetas = thetas
        self.__bandwidth = bandwidth
        self.__parametric_density = parametric_density

        self.__priori_target = priori_target
        self.__priori_fake = priori_fake

    def kernel_function(self, theta_1, theta_2):
        return torch.exp(-0.5*torch.norm(theta_1 - theta_2, dim = -1) / self.__bandwidth)

    def pdf_f(self, theta):
        applied_kernel = self.kernel_function(theta_1 = theta, theta_2 = self.__thetas)

        # print("theta: ", theta)
        pf_theta = self.__parametric_density.pdf_f(theta)
        # print("pf(theta): ", pf_theta)
        # # for t in self.__thetas:
        pesos = []
        for t in self.__thetas:
            pf_thetas_t = self.__parametric_density.pdf_f(t)
            peso = pf_theta / pf_thetas_t
            # print("theta: ", pf_theta)
            # print("theta_t: ", pf_thetas_t)
            # print("divisao: ", peso)
            pesos.append(pf_theta / pf_thetas_t)

        pesos = torch.as_tensor(pesos)
        # print(len(pesos))
        # print(pesos)

        # print(len(applied_kernel))
        # print(applied_kernel)
        mult = applied_kernel * pesos
        # print(mult)
        soma = mult.sum()
        # soma = torch.sum(applied_kernel)

        # print(soma / (self.__num_fake_samples * (self.__bandwidth ** self.__theta_dimension)))
        # print(applied_kernel)
        # print(pesos)
        # print(mult)
        # print(soma)
        # print((self.__num_fake_samples * (self.__bandwidth ** self.__theta_dimension)))
        return soma * (self.__num_fake_samples * (self.__bandwidth ** self.__theta_dimension))

    def log_f(self, theta):
        return torch.log(self.pdf_f(theta))

    def log_s(self, theta):
        # print(self.log_f(theta))
        # print(self.__priori_target.log(theta))
        # print(self.__priori_fake.log(theta))
        return self.__priori_target.log(theta) + self.log_f(theta) - self.__priori_fake.log(theta)

In [ ]:
def bandwidth_scott_logtheta(thetas: torch.Tensor, eps: float = 1e-12):
    x = torch.as_tensor(thetas).detach().flatten()
    x = torch.clamp(x, min=eps)
    z = torch.log(x)
    Tf = z.numel()
    d = 1
    std = torch.std(z, unbiased=True)
    return std * (Tf ** (-1.0 / (d + 4)))  # Tf^{-1/5}

def bandwidth_silverman_logtheta(thetas: torch.Tensor, eps: float = 1e-12):
    x = torch.as_tensor(thetas).detach().flatten()
    x = torch.clamp(x, min=eps)
    z = torch.log(x)
    Tf = z.numel()

    q75 = torch.quantile(z, 0.75)
    q25 = torch.quantile(z, 0.25)
    iqr = q75 - q25
    std = torch.std(z, unbiased=True)
    s = torch.minimum(std, iqr / 1.34)
    return 0.9 * s * (Tf ** (-1.0 / 5.0))


In [ ]:
# TODO: agora é rodar o modelo original com priori laplace e ver quais os resultados!!
bandwidth_1 = bandwidth_scott_logtheta(thetas=theta_samples)
bandwidth_2 = bandwidth_silverman_logtheta(thetas=theta_samples)

print(bandwidth_1)
sm_density = SemiParametricDistribution(
    parametric_density=parametric_density,
    priori_target = pi,
    priori_fake = pi_f,
    thetas=theta_samples,
    bandwidth=0.07
)
# sm_density.log_s(torch.as_tensor(0.09))

tensor(1.0447, dtype=torch.float64)


In [ ]:
estado_inicial = torch.randn((d,))
# TODO: gerar as amostras de theta usando Sampler
sampler_ps = Sampler(f_log_prob=sm_density.log_s)
samples_ps, burned, not_used = sampler_ps.get_samples(
    method="mh",
    estado_inicial=estado_inicial,
    use_log_pdf=True,
    n_amostras=10,
    burnin=0.1
)

Gerando amostras: 100%|██████████| 10/10 [00:12<00:00,  1.29s/it]


RuntimeError: stack expects a non-empty TensorList

In [ ]:
print(samples_ps.shape)